# Clase 2 — Programación cuántica con Cirq

En esta clase pasas de la teoría al código. Vas a escribir tu primer programa cuántico de verdad, con Cirq, la librería de Python que Google publicó para diseñar, simular y ejecutar circuitos cuánticos. Es la base sobre la que se construye TensorFlow Quantum, así que todo lo que veas hoy lo vas a reutilizar en cada clase siguiente.

**Objetivos de aprendizaje**

- Construir circuitos con `cirq.GridQubit` y `cirq.Circuit`
- Aplicar puertas de un qubit y de dos qubits
- Usar símbolos de `sympy` para crear circuitos entrenables

In [ ]:
!pip install -q cirq

## Los qubits viven en una rejilla

Arranca con una pregunta simple: ¿dónde vive un qubit dentro de Cirq? No es una idea abstracta flotando en el aire, tiene una posición concreta. La clase `cirq.GridQubit(fila, columna)` representa un qubit físico situado en una rejilla, igual que están distribuidos los qubits en un chip real como Sycamore.

Piensa en un tablero de ajedrez: cada pieza ocupa una casilla concreta, con su fila y su columna. Un `GridQubit` funciona igual. Cuando definas `cirq.GridQubit(0, 0)` y `cirq.GridQubit(0, 1)`, estás colocando dos qubits uno al lado del otro en la misma fila de la rejilla.

## Un circuito es una receta de pasos

Un `cirq.Circuit()` es la secuencia ordenada de puertas que aplicas a tus qubits. Piénsalo como una receta de cocina: la receta no funciona si mezclas los pasos, y un circuito cuántico tampoco. Añades cada puerta con `circuito.append(...)`, y Cirq las va organizando automáticamente en "momentos": un momento agrupa las puertas que se ejecutan a la vez, sobre qubits distintos, para aprovechar el paralelismo del hardware.

No necesitas gestionar tú ese agrupamiento, Cirq lo hace por ti. Tu trabajo es decidir qué puertas aplicas y en qué orden.

## El catálogo de puertas que más vas a usar

Estas son las puertas que vas a repetir en prácticamente cada circuito del curso:

- `cirq.H(qubit)`: Hadamard, crea superposición
- `cirq.X(qubit)`, `cirq.Y(qubit)`, `cirq.Z(qubit)`: rotaciones de 180 grados sobre cada eje, el equivalente cuántico de "voltear" el qubit
- `cirq.rx(angulo)`, `cirq.ry(angulo)`, `cirq.rz(angulo)`: rotaciones de un ángulo cualquiera sobre cada eje. Estas son las que usas para "entrenar" un circuito, porque el ángulo puede ser un parámetro ajustable
- `cirq.CNOT(control, objetivo)` y `cirq.CZ(control, objetivo)`: puertas de dos qubits que crean entrelazamiento
- Potencias fraccionarias, como `cirq.X(qubit)**0.5`: aplican solo "una parte" de la rotación. `cirq.X(qubit)**i` equivale a una rotación de `pi * i` radianes, una forma compacta de codificar un número real como ángulo de rotación (la usarás más adelante para convertir datos clásicos en estados cuánticos)

## Parámetros simbólicos con sympy

Aquí está la pieza que convierte un circuito cuántico en algo entrenable. Cirq se apoya en la librería `sympy` para definir puertas con un ángulo que todavía no conoces: un símbolo en vez de un número fijo.

Defines la puerta con un símbolo, por ejemplo `theta`, y más adelante un optimizador va probando distintos valores de `theta` hasta encontrar el que mejor resuelve tu problema. Esta idea, "parámetro simbólico que se ajusta durante el entrenamiento", es el equivalente cuántico de un peso en una red neuronal clásica. Guarda bien esta analogía, la vas a usar en cada clase de aquí en adelante.

## Simular y medir

Sin acceso a un ordenador cuántico real, simulas el circuito en tu propio ordenador con `cirq.Simulator()`. El método `.run(circuito, repetitions=N)` ejecuta el circuito N veces, porque cada ejecución individual solo te da un resultado (recuerda la clase 1), y te devuelve la estadística de resultados.

Vamos a verlo todo junto en el ejemplo completo.

In [ ]:
import cirq
import sympy

# Dos qubits en una rejilla de una sola fila
q0, q1 = cirq.GridQubit(0, 0), cirq.GridQubit(0, 1)

# Un símbolo: un ángulo que todavía no tiene valor numérico
theta = sympy.Symbol("theta")

circuito = cirq.Circuit()

# Ponemos el primer qubit en superposición
circuito.append(cirq.H(q0))

# Entrelazamos los dos qubits
circuito.append(cirq.CNOT(q0, q1))

# Rotamos el segundo qubit un ángulo "theta" sobre el eje Y (parámetro entrenable)
circuito.append(cirq.ry(theta).on(q1))

# Medimos los dos qubits
circuito.append(cirq.measure(q0, q1, key="resultado"))

print(circuito)

Fíjate en el diagrama que acabas de imprimir: ves las dos líneas de los qubits y, en orden, la Hadamard, el CNOT y la rotación con el símbolo `theta` todavía sin valor. El circuito está completo, pero no puedes simularlo tal cual, porque `theta` no es un número. Necesitas un `cirq.ParamResolver` que le asigne un valor concreto.

In [ ]:
# Para simular, sustituimos el símbolo "theta" por un valor concreto
simulador = cirq.Simulator()
resolver = cirq.ParamResolver({"theta": 0.5})
resultado = simulador.run(circuito, resolver, repetitions=500)

print(resultado.histogram(key="resultado"))

El `cirq.ParamResolver` es el mecanismo que sustituye el símbolo `theta` por un número concreto justo antes de simular. Guarda bien este patrón: circuito con símbolos más resolver que asigna valores. Es exactamente el que usa TensorFlow Quantum por debajo cuando entrena un modelo, solo que en vez de asignar el valor a mano, un optimizador lo va ajustando automáticamente, paso a paso, hasta minimizar un error.

## Resumen de puntos clave

- `cirq.GridQubit` sitúa un qubit en una posición concreta de una rejilla, como en un chip real
- `cirq.Circuit()` y `.append(...)` construyen la secuencia de puertas del circuito
- Las puertas de rotación (`rx`, `ry`, `rz`) aceptan símbolos de `sympy` como ángulo, lo que convierte el circuito en "entrenable"
- Las potencias fraccionarias de una puerta (`cirq.X(q)**i`) codifican un número real como rotación
- `cirq.Simulator()` ejecuta el circuito; `cirq.ParamResolver` sustituye los símbolos por valores concretos antes de simular